Gauntlet #4: The Corrupted Sensor Data Challenge

In [1]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Generate realistic temperatures: average 15°C, with daily and hourly variations
days = 365
hours = 24
base = 15.0
daily_variation = 10 * np.sin(np.linspace(0, 2*np.pi, days))[:, np.newaxis]
hourly_variation = 5 * np.sin(np.linspace(0, 2*np.pi, hours))[np.newaxis, :]
noise = np.random.randn(days, hours) * 3

temps = base + daily_variation + hourly_variation + noise
temps = temps.astype(np.float32)

# Introduce corruption
# -999.0 placeholders (5% of data)
mask_999 = np.random.rand(*temps.shape) < 0.05
temps[mask_999] = -999.0

# Extreme values (2% of data)
mask_extreme = np.random.rand(*temps.shape) < 0.02
temps[mask_extreme] = np.random.choice([-100.0, 80.0, np.nan, np.inf], size=mask_extreme.sum())

# Save to binary file
temps.tofile('sensor_data.dat')
print("File 'sensor_data.dat' created. Shape:", temps.shape)

File 'sensor_data.dat' created. Shape: (365, 24)


In [2]:
import numpy.ma as ma
shape = (365, 24)
dtype = np.float32
arr = np.memmap('sensor_data.dat', dtype=dtype, mode='r+', shape=shape)

invalid_mask = [
                (arr == -999) |
                (arr > 60) |
                (arr < -50) |
                np.isnan(arr) |
                np.isinf(arr)
                ]

masked = ma.MaskedArray(arr, mask=invalid_mask)
print("Masked array created. Valid data count: ", masked.count())

Masked array created. Valid data count:  8182


In [3]:
overall_avg = masked.mean()
print(f"Average temperature for the entier year: {overall_avg:.2f}")

Average temperature for the entier year: 14.96


In [4]:
# Reshape to (12, 30, 24) – slice off the extra 5 days
monthly_data = masked[:360].reshape(12, 30, 24)

# Mean over days and hours -> shape (12,)
monthly_avg = monthly_data.mean(axis=(1, 2))
print("Monthly averages:")
for m, avg in enumerate(monthly_avg):
    print(f"  Month {m+1:2d}: {avg:.2f}°C")

Monthly averages:
  Month  1: 17.40°C
  Month  2: 22.05°C
  Month  3: 24.61°C
  Month  4: 24.68°C
  Month  5: 22.28°C
  Month  6: 17.71°C
  Month  7: 12.87°C
  Month  8: 8.18°C
  Month  9: 5.52°C
  Month 10: 5.25°C
  Month 11: 7.49°C
  Month 12: 11.79°C


In [5]:
# Get a filled version with -inf for masked values (so they are never max)
valid_only = masked.filled(-np.inf)
max_temp = valid_only.max()
max_location = np.unravel_index(valid_only.argmax(), valid_only.shape)
print(f"Hottest hour: {max_temp:.2f}°C at Day {max_location[0]}, Hour {max_location[1]}")

Hottest hour: 38.44°C at Day 67, Hour 7


In [6]:
# Work with the 360-day reshaped array (12 months, 30 days, 24 hours)
monthly_masked = masked[:360].reshape(12, 30, 24)

# Compute mean over days (axis=1) -> shape (12, 24)
# keepdims=True to broadcast back to (12, 30, 24)
monthly_hour_mean = monthly_masked.mean(axis=1, keepdims=True)

# Fill masked values in monthly_masked with the corresponding mean
filled_monthly = monthly_masked.filled(monthly_hour_mean)

# Now we need to put the filled data back into the original shape (365,24)
# Create a new array for the full year, starting with original data
cleaned = arr.copy()  # This copies into RAM! We'll fix with memmap write.

# Better: write directly to a new memmap file, chunk by chunk
clean_file = np.memmap('sensor_data_clean.dat', dtype=dtype, mode='w+', shape=shape)

# First 360 days: use filled monthly data
clean_file[:360] = filled_monthly.reshape(360, 24)

# Last 5 days: fill with the average of the corresponding hour across all months?
# For simplicity, fill with the global hourly average from the 360 days
hourly_avg_global = monthly_masked.mean(axis=(0,1))  # shape (24,)
for day in range(360, 365):
    clean_file[day] = hourly_avg_global

# Flush and close
clean_file.flush()
del clean_file
print("Cleaned data saved to 'sensor_data_clean.dat'")

Cleaned data saved to 'sensor_data_clean.dat'


In [7]:
# Quick check: no invalid values should remain
clean = np.memmap('sensor_data_clean.dat', dtype=dtype, mode='r', shape=shape)
invalid_remaining = (clean == -999.0) | (clean > 60.0) | (clean < -50.0) | np.isnan(clean) | np.isinf(clean)
print("Invalid values remaining:", invalid_remaining.sum())

Invalid values remaining: 0
